# Proyecto Integrador de Ciencia de Datos — **Fase 1**
## Inasistencia escolar en la adolescencia paraguaya: magnitud, motivos declarados y brechas territoriales (2022–2024)

---

| | |
|---|---|
| **Asignatura** | Data Science |
| **Fase** | 1 de 3 — Ponderación 30 % |
| **Integrantes** | `Apellido1, Nombre — Legajo` · `Apellido2, Nombre — Legajo` · `Apellido3, Nombre — Legajo` |
| **Fecha de entrega** | Lunes 15 de septiembre |
| **Fuente** | Encuesta Permanente de Hogares Continua (EPHC) — Dirección General de Estadística, Encuestas y Censos (DGEEC), Paraguay |
| **Archivos** | `REG02_EPHC_ANUAL_2022.csv` · `REG02_EPHC_ANUAL_2023.csv` · `REG02_EPHC_ANUAL_2024.csv` (registro de personas, Sección 4 — Educación) |

---

## Por qué se cambió la fuente de datos

La propuesta original de este proyecto preguntaba por los **factores institucionales** (servicios básicos,
infraestructura) asociados al **abandono escolar** entre 2015 y 2024, usando registros administrativos del
Ministerio de Educación y Ciencias (MEC). Al inventariar los archivos efectivamente disponibles en el portal
del MEC se comprobó que:

1. Los datasets de matrícula del MEC cubren un único año lectivo (2023) en el corte descargado.
2. **Ningún dataset del MEC contiene una variable de abandono, deserción o repitencia.**
3. Las variables de servicios básicos existen (`servicios_basicos` de FONACIDE), pero atadas a establecimientos
   priorizados para inversión, no al universo completo de instituciones ni a una serie temporal comparable.

La **EPHC de la DGEEC**, en cambio, sí trae —todos los años, con metodología estable— la pregunta **ED08**
(si la persona asiste actualmente) y, crucialmente, **ED10**: *"¿Por qué no asiste o dejó de asistir?"*, con
el motivo declarado por el propio hogar. Es la variable de abandono que la propuesta original necesitaba y
que el MEC no publica en datos abiertos. El costo de este cambio es real y se declara sin rodeos: la unidad
de análisis deja de ser el *establecimiento* y pasa a ser la *persona dentro de un hogar*, con un diseño
muestral complejo que exige usar el factor de expansión (`FEX`) en todo cálculo descriptivo.

> Si en el futuro se consigue el dataset `servicios_basicos` del MEC a nivel de establecimiento y con
> cobertura nacional, se podría intentar una estimación de disponibilidad de servicios **por departamento**
> y cruzarla con las tasas de inasistencia calculadas aquí. Esa vía queda anotada como trabajo futuro.

### Cómo usar este notebook

1. Los tres CSV (`REG02_EPHC_ANUAL_2022.csv`, `_2023.csv`, `_2024.csv`) van en `data/raw/`, sin modificar.
2. `Kernel → Restart & Run All`. El notebook corre de principio a fin y regenera
   `data/processed/dataset_fase1.parquet`.
3. Los bloques 🟡 **COMPLETAR** exigen redacción propia del grupo: el código produce la evidencia, la
   interpretación es de ustedes y es lo que se evalúa.
4. Ninguna transformación se hace a mano. Todo queda documentado en la **bitácora de limpieza** (Etapa 3.6).

---

## 0. Configuración y reproducibilidad

In [ ]:
# ============================================================================
# Imports y configuración global
# ============================================================================
import re
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore', category=FutureWarning)

SEMILLA = 42
np.random.seed(SEMILLA)

RAIZ     = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DIR_RAW  = RAIZ / 'data' / 'raw'
DIR_PROC = RAIZ / 'data' / 'processed'
DIR_FIG  = RAIZ / 'output' / 'figuras'
DIR_TAB  = RAIZ / 'output' / 'tablas'
for _d in (DIR_RAW, DIR_PROC, DIR_FIG, DIR_TAB):
    _d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.figsize': (10, 5), 'figure.dpi': 110, 'savefig.dpi': 200,
                     'savefig.bbox': 'tight', 'axes.titlesize': 13, 'axes.titleweight': 'bold',
                     'axes.labelsize': 11, 'font.size': 10})
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 170)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

print('Python:', sys.version.split()[0], '| pandas:', pd.__version__, '| numpy:', np.__version__)
print('Raíz  :', RAIZ)

In [ ]:
# ============================================================================
# CONFIG — único lugar a tocar si cambian los archivos o los códigos de variable
# ============================================================================

ARCHIVOS_EPHC = {
    2022: 'REG02_EPHC_ANUAL_2022.csv',
    2023: 'REG02_EPHC_ANUAL_2023.csv',
    2024: 'REG02_EPHC_ANUAL_2024.csv',
}

# Columnas del registro de personas (REG02) que se necesitan para este proyecto.
# El archivo real trae >200 columnas; sólo se cargan las que se van a usar.
COLUMNAS_NECESARIAS = [
    'UPM', 'NVIVI', 'NHOGA', 'AÑO', 'DPTO', 'AREA', 'L02',
    'P02',      # edad
    'P06',      # sexo (1 = Hombre, 6 = Mujer)
    'P04',      # es miembro del hogar
    'ED03',     # ¿asistió alguna vez a enseñanza formal?
    'ED08',     # ¿asiste actualmente? (19 = No asiste)
    'ED09',     # sector de la institución (1 Oficial, 2 Privado, 3 Privado subvencionado)
    'ED10',     # motivo de no asistencia / abandono
    'FEX.2022', # factor de expansión muestral (¡obligatorio para cualquier descriptivo!)
]

# Rango etario de interés: 3er ciclo de la EEB + educación media en Paraguay
# (edad teórica 12 a 17 años). Se declara aquí y se valida empíricamente en la Etapa 2.
EDAD_MIN, EDAD_MAX = 12, 17

# --- Diccionarios de categorías, tomados del diccionario oficial de variables ---
MAPA_AREA = {1: 'Urbana', 6: 'Rural'}
MAPA_SEXO = {1: 'Hombre', 6: 'Mujer'}
MAPA_ED09_SECTOR = {1: 'Oficial', 2: 'Privado', 3: 'Privado subvencionado', 9: 'NR'}

MAPA_DPTO = {
    0: 'Capital', 1: 'Concepción', 2: 'San Pedro', 3: 'Cordillera', 4: 'Guairá',
    5: 'Caaguazú', 6: 'Caazapá', 7: 'Itapúa', 8: 'Misiones', 9: 'Paraguarí',
    10: 'Alto Paraná', 11: 'Central', 12: 'Ñeembucú', 13: 'Amambay', 14: 'Canindeyú',
    15: 'Presidente Hayes', 16: 'Boquerón', 17: 'Alto Paraguay',
}

MAPA_ED10_MOTIVO = {
    1: 'Sin recursos en el hogar', 2: 'Necesidad de trabajar',
    3: 'Muy costosos materiales y matrículas', 4: 'No tiene edad adecuada',
    5: 'Considera que terminó los estudios', 6: 'No existe institución cercana',
    7: 'Institución cercana muy mala', 8: 'El centro educativo cerró',
    9: 'El docente no asiste con regularidad', 10: 'Institución no ofrece escolaridad completa',
    11: 'Requiere educación especial', 12: 'Por enfermedad',
    13: 'Debe hacer labores en el hogar', 14: 'Motivos familiares',
    15: 'No quiere estudiar', 16: 'Asiste a formación profesional/vocacional',
    17: 'Servicio militar', 18: 'Otra razón', 99: 'NR',
}

# ED08: 19 = 'No asiste'; el resto de los códigos (1-18) corresponden a 'Sí, <nivel>'
COD_NO_ASISTE = 19

print('Configuración cargada. Archivos esperados:')
for anio, nombre in ARCHIVOS_EPHC.items():
    ruta = DIR_RAW / nombre
    print(f'  {anio}: {nombre:32} {"✅ encontrado" if ruta.exists() else "⬜ FALTA"}')

---

# Etapa 1 — Comprensión del problema

## 1.1. Contexto del problema

La **Dirección General de Estadística, Encuestas y Censos (DGEEC)** de Paraguay releva anualmente la
**Encuesta Permanente de Hogares Continua (EPHC)**, una encuesta muestral probabilística a hogares que
releva, entre otras dimensiones, la situación educativa de cada miembro del hogar: si asiste actualmente a
una institución de enseñanza formal, a qué nivel, en qué sector, y —si no asiste o dejó de asistir— **por
qué**. La encuesta usa un diseño muestral complejo con unidades primarias de muestreo (UPM) y un factor de
expansión (`FEX`) que permite generalizar los resultados de la muestra a la población total del país.

La **decisión real en juego** es la misma que motivó la propuesta original del proyecto: cómo priorizar la
inversión y las políticas de retención escolar. La diferencia es la fuente: en vez de partir del registro
administrativo de establecimientos del MEC, este proyecto parte de la **voz del hogar**, que declara
directamente por qué un adolescente no está en la escuela. Esa información no la tiene el MEC —un
establecimiento no puede reportar por qué un chico que nunca llegó a inscribirse dejó de estudiar— y es
exactamente la que permite distinguir entre causas económicas, familiares, de oferta educativa o de interés
personal.

Los usuarios potenciales del análisis son el MEC (para políticas de retención), el Ministerio de Desarrollo
Social (para programas de transferencias condicionadas), las gobernaciones y organizaciones que monitorean
la equidad educativa territorial.

## 1.2. Pregunta de investigación

> **¿En qué medida la zona de residencia (urbana/rural) y el departamento se asocian con la tasa de
> inasistencia escolar de los adolescentes de 12 a 17 años en Paraguay, y qué motivos declaran los hogares
> para explicar esa inasistencia, entre 2022 y 2024?**

La pregunta cumple los tres criterios exigidos: se responde con **evidencia** (la EPHC, encuesta oficial de
la DGEEC) y no con opinión; está **acotada** en tiempo (2022–2024, los tres años con archivo disponible),
en edad (12 a 17 años, edad teórica de 3.er ciclo y educación media) y en alcance (zona, departamento y
motivo declarado); y **admite respuesta negativa**: es posible que, controlando por departamento, la
diferencia urbano/rural no resulte relevante, o que el motivo principal de inasistencia no varíe entre
zonas.

## 1.3. Objetivo general

**Analizar** la asociación entre la zona de residencia, el departamento y la tasa de inasistencia escolar
de los adolescentes de 12 a 17 años en Paraguay entre 2022 y 2024, caracterizando los motivos de
inasistencia declarados por los hogares.

## 1.4. Objetivos específicos

| N.° | Objetivo específico | Fase | Sección |
|:--:|---|:--:|---|
| **OE1** | **Construir** un panel 2022–2024 de personas de 12 a 17 años a partir de los tres registros anuales de la EPHC, con las variables de asistencia, motivo, zona, departamento y factor de expansión correctamente aplicado. | Fase 1 | Etapas 2 y 3 |
| **OE2** | **Describir** la evolución de la tasa de inasistencia escolar y de sus motivos declarados, por zona, departamento y sexo, entre 2022 y 2024. | Fase 1 | Etapas 4 y 5 |
| **OE3** | **Determinar** si la tasa de inasistencia difiere de forma estadísticamente significativa entre zonas urbana y rural, y entre departamentos, y si el motivo declarado se asocia con la zona. | **Fase 2 (inferencial)** | Contrastes de hipótesis |
| **OE4** | **Construir** un modelo de clasificación que estime la probabilidad de inasistencia escolar de un adolescente a partir de su zona, departamento, edad y sexo, comparándolo contra una línea base. | **Fase 3 (predictiva)** | Modelado y evaluación |
| **OE5** | **Comunicar** los hallazgos y sus limitaciones —en particular las propias de trabajar con una encuesta muestral— en términos utilizables por responsables de política educativa. | Cierre | Informe y presentación |

OE1 y OE2 se ejecutan íntegramente en esta fase. Las hipótesis preliminares de la sección 5.3 son el insumo
directo de OE3 (Fase 2), y la selección de variables de la Etapa 3 define el espacio de atributos de OE4
(Fase 3).

## 1.5. Justificación de la relevancia

Paraguay ha ampliado su cobertura educativa en las últimas décadas, pero el 3.er ciclo de la escolar básica
y la educación media siguen siendo el tramo donde más adolescentes salen del sistema. Las estadísticas
agregadas de matrícula no explican **por qué** ocurre esa salida; la EPHC sí lo pregunta directamente. Un
análisis que distinga si la inasistencia se debe mayoritariamente a razones económicas, familiares, de
oferta educativa insuficiente o de decisión personal tiene una implicancia de política completamente
distinta en cada caso: transferencias monetarias, programas de cuidado familiar, ampliación de la oferta
escolar, o estrategias de motivación y orientación vocacional, respectivamente.

Además, al tratarse de una encuesta con tres años consecutivos, el proyecto puede documentar si la brecha
urbano/rural se mantiene, se amplía o se reduce en el período reciente —algo que un corte único no permite.

## 1.6. Alcance y limitaciones previstas

**Dentro del alcance**

- Personas de **12 a 17 años** registradas en el hogar (`P04` = miembro del hogar).
- Período **2022–2024** (los tres años con archivo disponible).
- Variables de **asistencia actual** (`ED08`), **motivo de inasistencia** (`ED10`), **sector** (`ED09`),
  **zona** (`AREA`), **departamento** (`DPTO`), **sexo** (`P06`) y **edad** (`P02`).
- Todo estadístico descriptivo se calcula **ponderado por el factor de expansión** (`FEX.2022`), como exige
  el diseño muestral de la encuesta.

**Explícitamente fuera**

- **Características del establecimiento** (infraestructura, servicios básicos, planta docente): la EPHC no
  identifica el establecimiento al que asiste o asistía la persona; esa información seguiría el registro
  administrativo del MEC. No se fuerza un cruce que la propia estructura de los datos no permite.
- **Seguimiento de cohorte.** La EPHC es una encuesta de **corte transversal repetido**, no un panel: no se
  entrevista a los mismos hogares cada año. "2022–2024" describe tres fotografías comparables, no la
  trayectoria de las mismas personas en el tiempo.
- **Inferencia causal.** Las asociaciones que se documenten son observacionales; nunca se afirma que la
  zona o el departamento *causen* la inasistencia, sólo que se *asocian* con ella.
- Niveles educativos fuera del rango 12-17 años (inicial, primaria temprana, superior).

**Limitaciones de diseño que se declaran desde ahora**

1. **Diseño muestral complejo.** La EPHC tiene estratificación y conglomerados (UPM); los cálculos de este
   proyecto usan el factor de expansión para las estimaciones puntuales, pero **no** corrigen los errores
   estándar por el efecto de diseño (eso requeriría el paquete de replicación de la DGEEC, fuera del alcance
   de la materia). Los intervalos de confianza de la Fase 2 deben leerse con esa salvedad.
2. **Tamaño de submuestra por departamento.** Algunos departamentos pequeños (Alto Paraguay, Boquerón,
   Ñeembucú) tienen pocas observaciones muestrales; sus estimaciones tendrán mayor error y se señalará
   explícitamente.
3. **El motivo de inasistencia es autorreportado** por el hogar, no verificado independientemente.
4. **Cobertura territorial incompleta en el Chaco.** Se verifica empíricamente en la Etapa 2.6 si Boquerón
   y Alto Paraguay están representados en la muestra de los tres años; si no lo están, los resultados de
   este proyecto no podrán extenderse a esa región y así se declarará en la conclusión.

---

# Etapa 2 — Comprensión y selección de los datos

In [ ]:
# ============================================================================
# 2.1 — Inventario de los archivos disponibles
# ============================================================================
filas = []
for anio, nombre in ARCHIVOS_EPHC.items():
    ruta = DIR_RAW / nombre
    if not ruta.exists():
        filas.append({'archivo': nombre, 'anio': anio, 'estado': 'NO ENCONTRADO'})
        continue
    with open(ruta, encoding='utf-8-sig') as f:
        n_cols = len(f.readline().split(';'))
    n_filas = sum(1 for _ in open(ruta, encoding='utf-8-sig')) - 1
    filas.append({'archivo': nombre, 'anio': anio, 'tamano_mb': round(ruta.stat().st_size/1024**2, 1),
                  'filas': n_filas, 'columnas_totales': n_cols, 'estado': 'OK'})

inventario = pd.DataFrame(filas)
display(inventario)
inventario.to_csv(DIR_TAB / 'inventario_archivos.csv', index=False)

### 2.1.1. Descripción del inventario

Los tres archivos son el **registro de personas (REG02)** de la EPHC anual, uno por año. Cada fila es una
persona dentro de un hogar dentro de una vivienda. Comparten exactamente la misma estructura de columnas
(213 en los tres casos), lo que permite concatenarlos directamente en un panel sin necesidad de mapear
nombres de campo entre años —a diferencia de lo que suele ocurrir con fuentes administrativas.

| Archivo | Año | Contenido | Unidad de análisis | Clave |
|---|:--:|---|---|---|
| `REG02_EPHC_ANUAL_2022.csv` | 2022 | Registro de personas, todas las secciones (incl. Educación) | persona | `UPM`+`NVIVI`+`NHOGA`+`L02` |
| `REG02_EPHC_ANUAL_2023.csv` | 2023 | ídem | persona | ídem |
| `REG02_EPHC_ANUAL_2024.csv` | 2024 | ídem | persona | ídem |

In [ ]:
# ============================================================================
# 2.2 — Carga de los archivos
# ============================================================================
# ⚠️ Detalle crítico de formato: el archivo usa ';' como separador de columnas
# y ',' como separador DECIMAL (formato numérico latinoamericano). Si se ignora
# el parámetro decimal=',', el factor de expansión FEX se lee como texto y
# cualquier .sum() posterior concatena strings en vez de sumar números,
# produciendo un resultado sin sentido y sin ningún error visible.

tablas_anuales = {}
for anio, nombre in ARCHIVOS_EPHC.items():
    ruta = DIR_RAW / nombre
    df_anio = pd.read_csv(ruta, sep=';', encoding='utf-8-sig', decimal=',',
                          usecols=lambda c: c in COLUMNAS_NECESARIAS, low_memory=False)
    df_anio['anio_encuesta'] = anio   # se usa el año de archivo como ancla temporal
    tablas_anuales[anio] = df_anio
    print(f'{nombre:32} -> {df_anio.shape[0]:>7,} filas × {df_anio.shape[1]} columnas   '
          f'FEX dtype: {df_anio["FEX.2022"].dtype}')

crudo = pd.concat(tablas_anuales.values(), ignore_index=True)
print(f'\nPanel unificado 2022–2024: {crudo.shape[0]:,} filas × {crudo.shape[1]} columnas')
crudo.head()

In [ ]:
# ============================================================================
# 2.3 — Diccionario de datos (reconstruido por el grupo)
# ============================================================================
def diccionario_datos(df: pd.DataFrame, max_cat: int = 6) -> pd.DataFrame:
    filas = []
    for col in df.columns:
        s, nn = df[col], df[col].dropna()
        if pd.api.types.is_numeric_dtype(s) and nn.nunique() > max_cat:
            rango = f'[{nn.min():,.2f} — {nn.max():,.2f}]' if len(nn) else '—'
        else:
            cats = nn.astype(str).value_counts().index[:max_cat].tolist()
            extra = '' if nn.nunique() <= max_cat else f' … (+{nn.nunique()-max_cat})'
            rango = ', '.join(cats) + extra
        filas.append({'campo': col, 'tipo_detectado': str(s.dtype), 'tipo_correcto': '',
                      'descripcion': '', 'unidad': '', 'n_unicos': s.nunique(dropna=True),
                      'rango_o_categorias': rango, 'pct_faltantes': round(100*s.isna().mean(), 2)})
    return pd.DataFrame(filas)

dicc_original = diccionario_datos(crudo)
dicc_original.to_csv(DIR_TAB / 'diccionario_datos_original.csv', index=False)
display(dicc_original)
print('🟡 Completar a mano descripcion / tipo_correcto / unidad de cada campo antes de pegarla en el informe.')
print('   Referencia oficial: diccionario_EPHC_ANUAL_2023.xls, sección "POBLACIÓN" y "EDUCACIÓN".')

In [ ]:
# ============================================================================
# 2.4 — Unidad de análisis y clave: verificación EMPÍRICA de unicidad
# ============================================================================
CLAVE_PERSONA = ['UPM', 'NVIVI', 'NHOGA', 'L02', 'anio_encuesta']

dup = int(crudo.duplicated(subset=CLAVE_PERSONA).sum())
print(f'Clave candidata: {CLAVE_PERSONA}')
print(f'Filas totales               : {len(crudo):,}')
print(f'Combinaciones únicas de clave: {crudo[CLAVE_PERSONA].drop_duplicates().shape[0]:,}')
print(f'Filas duplicadas por esa clave: {dup:,}')
print(f'¿Es clave primaria dentro de cada año? {"✅ Sí" if dup == 0 else "❌ No — revisar"}')

# La unidad de análisis declarada en la Etapa 1.6 es la persona-año.
print('\nUnidad de análisis adoptada: PERSONA dentro de un HOGAR, observada en un AÑO de encuesta',
      '(corte transversal repetido, no panel de las mismas personas).')

In [ ]:
# ============================================================================
# 2.5 — Perfilado inicial
# ============================================================================
print(f'Dimensiones: {crudo.shape[0]:,} filas × {crudo.shape[1]} columnas')
print(f'Memoria    : {crudo.memory_usage(deep=True).sum()/1024**2:,.1f} MB\n')

perfil = pd.DataFrame({
    'tipo_detectado': crudo.dtypes.astype(str),
    'n_unicos': crudo.nunique(dropna=True),
    'faltantes_%': (100*crudo.isna().mean()).round(2),
}).sort_values('faltantes_%', ascending=False)
display(perfil)

print('Distribución de edad (P02):')
display(crudo.P02.describe(percentiles=[.05, .25, .5, .75, .95]))

print('\nValores observados de ED08 (asistencia) — códigos crudos:')
print(crudo.ED08.value_counts(dropna=False).sort_index())

In [ ]:
# ============================================================================
# 2.6 — Evaluación de calidad
# ============================================================================
chequeos = []
chequeos.append(('Completitud global', f'{100*(1-crudo.isna().mean().mean()):.2f} % de celdas informadas'))
chequeos.append(('Cobertura temporal', f'{sorted(crudo.anio_encuesta.unique())}'))
chequeos.append(('Filas duplicadas exactas', f'{crudo.duplicated().sum():,}'))
chequeos.append(('Edad fuera de rango plausible (< 0 o > 110)',
                 f'{int(((crudo.P02 < 0) | (crudo.P02 > 110)).sum()):,}'))
chequeos.append(('Población total ponderada por año (FEX)',
                 crudo.groupby('anio_encuesta')['FEX.2022'].sum().round(0).astype(int).to_dict()))
chequeos.append(('Códigos de ED08 fuera del diccionario oficial (1-19, 99, blanco)',
                 sorted(set(crudo.ED08.dropna().unique()) - set(range(1, 20)) - {99})))

# --- Cobertura territorial: ¿aparecen los 17 departamentos + Capital? -------
dptos_observados = sorted(int(d) for d in crudo.DPTO.dropna().unique())
dptos_esperados  = set(range(0, 18))
dptos_ausentes   = sorted(dptos_esperados - set(dptos_observados))
chequeos.append(('Departamentos observados en la muestra (códigos)', dptos_observados))
chequeos.append(('Departamentos NUNCA muestreados en 2022-2024', dptos_ausentes))
display(pd.DataFrame(chequeos, columns=['Chequeo', 'Resultado']))

if dptos_ausentes:
    nombres_ausentes = [MAPA_DPTO.get(d, d) for d in dptos_ausentes]
    print(f'\n⚠️  {nombres_ausentes} no aparecen en NINGUNO de los tres años.')
    print('    Esto es sistemático, no un error de cruce: la EPHC no releva esos departamentos')
    print('    con la periodicidad y desagregación que sí tiene el resto del país. Se declara como')
    print('    límite de cobertura territorial en la Etapa 1.6 y en la conclusión de la Fase 1: ningún')
    print('    resultado de este proyecto puede extenderse a la región del Chaco central/norte.')

### 2.7. Evaluación de calidad — lectura

**Completitud.** El registro de personas está prácticamente completo en las variables de identificación,
zona y departamento; los códigos en blanco de `ED08`/`ED09`/`ED10` no son datos faltantes en sentido
estricto, sino personas **fuera del universo de la pregunta** (por ejemplo, adultos a quienes no corresponde
preguntarles si asisten a educación media). Ese universo se delimita en la Etapa 3 filtrando por edad.

**Consistencia.** La estructura de columnas es idéntica en los tres años, lo que evita el problema típico
de fuentes administrativas (nombres de campo que cambian entre períodos). El punto de fricción real fue de
**formato**: el separador decimal en coma exige `decimal=','` al leer, verificado en la Etapa 2.2.

**Exactitud aparente.** La suma del factor de expansión por año debe aproximarse a la población total de
Paraguay (del orden de 6-7 millones); si ese número no aparece en el orden de magnitud correcto, es señal
de que el factor se está leyendo mal (ver nota de la Etapa 2.2).

**Actualidad.** Los tres años están completos y son los más recientes publicados por la DGEEC al momento
de la descarga.

**Trazabilidad.** La fuente (DGEEC, EPHC anual) y el diccionario oficial de variables permiten auditar cada
código contra la documentación metodológica publicada.

### 2.8. Selección justificada

| Elemento | Decisión | Justificación | Objetivo vinculado |
|---|:--:|---|:--:|
| Sección Educación (`ED01`–`ED11GH1A`) | **Se conserva parcialmente** (`ED03`, `ED08`, `ED09`, `ED10`) | Son las variables de asistencia, sector y motivo que responden la pregunta | OE1–OE4 |
| `ED0504` (nivel más alto aprobado) y `ED06C` (título obtenido) | **Se descarta** | Describen la trayectoria educativa completa, no la asistencia actual ni el motivo de inasistencia | — |
| `ED11F1`, `ED11GH1` (merienda/almuerzo escolar) | **Se descarta de esta fase** | Es un programa social distinto al fenómeno de inasistencia; podría explorarse como variable de control en la Fase 3 | (posible OE4) |
| Todas las secciones de empleo, salud e ingresos (`A01`–`C…`) | **Se descarta** | Fuera del alcance declarado en la Etapa 1.6; ninguna hipótesis del proyecto las requiere | — |
| Personas fuera de 12–17 años | **Se descarta para el análisis principal** | La pregunta está acotada a esa edad teórica (Etapa 1.6); se conserva el archivo completo por si se decide ampliar el rango más adelante | OE1–OE4 |

### 2.9. Documentación de la fuente

| Ítem | Valor |
|---|---|
| Fuente | Dirección General de Estadística, Encuestas y Censos (DGEEC), Paraguay |
| Encuesta | Encuesta Permanente de Hogares Continua (EPHC), rondas anuales 2022, 2023 y 2024 |
| Archivos | `REG02_EPHC_ANUAL_2022.csv`, `REG02_EPHC_ANUAL_2023.csv`, `REG02_EPHC_ANUAL_2024.csv` |
| Diccionario de variables | `diccionario_EPHC_ANUAL_2023.xls` (hoja `EPHC 2023`) |
| Licencia | Microdatos de uso público de la DGEEC |
| Unidad de medida del factor de expansión | Personas representadas en la población (`FEX.2022`) |
| Responsable de la descarga | 🟡 *(completar)* |
| Fecha de descarga | 🟡 *(completar)* |

---

# Etapa 3 — Limpieza y transformación

> **Regla no negociable:** toda transformación es reproducible por código desde los archivos originales.
> Nada se edita a mano. Cada decisión queda registrada en la bitácora (sección 3.6).

In [ ]:
# ============================================================================
# Bitácora de limpieza
# ============================================================================
class Bitacora:
    def __init__(self):
        self.registros = []
    def anotar(self, etapa, decision, afectados, fundamento, antes=None, despues=None):
        self.registros.append({'n': len(self.registros)+1, 'etapa': etapa, 'decision': decision,
                               'registros_afectados': afectados, 'filas_antes': antes,
                               'filas_despues': despues, 'fundamento': fundamento})
        print(f'[{len(self.registros):02d}] {etapa} — {decision} ({afectados:,})')
    def tabla(self):
        return pd.DataFrame(self.registros)

bitacora = Bitacora()
df = crudo.copy()
FILAS_INICIALES = len(df)
print(f'Punto de partida: {FILAS_INICIALES:,} filas (todas las personas, todas las edades, 2022-2024)')

In [ ]:
# ============================================================================
# 3.1 — Normalización de tipos y de categorías
# ============================================================================

# Renombrado a nombres de trabajo legibles (se conserva la equivalencia en el diccionario)
df = df.rename(columns={
    'AÑO': 'anio_dato', 'DPTO': 'cod_departamento', 'AREA': 'cod_zona', 'P02': 'edad',
    'P06': 'cod_sexo', 'ED08': 'cod_asistencia', 'ED09': 'cod_sector', 'ED10': 'cod_motivo',
    'FEX.2022': 'factor_expansion',
})
bitacora.anotar('3.1 Tipos', 'Renombrado de columnas a nombres de trabajo legibles', df.shape[1],
                'Los códigos crudos (P02, ED08, …) no son autoexplicativos; se documenta la equivalencia en el diccionario de datos.')

for col in ['cod_departamento', 'cod_zona', 'cod_sexo', 'cod_asistencia', 'cod_sector', 'cod_motivo', 'edad']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['zona'] = df.cod_zona.map(MAPA_AREA)
df['sexo'] = df.cod_sexo.map(MAPA_SEXO)
df['departamento'] = df.cod_departamento.map(MAPA_DPTO)
df['sector'] = df.cod_sector.map(MAPA_ED09_SECTOR)
df['motivo_inasistencia'] = df.cod_motivo.map(MAPA_ED10_MOTIVO)
bitacora.anotar('3.1 Tipos', 'Mapeo de códigos numéricos a etiquetas legibles (zona, sexo, departamento, sector, motivo)',
                len(df), 'Los códigos numéricos no son interpretables sin el diccionario oficial; se decodifican aquí una sola vez.')

n_sin_dpto = int(df.departamento.isna().sum())
if n_sin_dpto:
    print(f'⚠️ {n_sin_dpto} filas con código de departamento fuera de 0–17 (no mapeado). Revisar en 2.6.')

display(df[['anio_encuesta', 'edad', 'zona', 'sexo', 'departamento', 'sector', 'cod_asistencia']].head())

In [ ]:
# ============================================================================
# 3.2 — Delimitación del universo de análisis: personas de 12 a 17 años
# ============================================================================
antes = len(df)
n_fuera_edad = int((~df.edad.between(EDAD_MIN, EDAD_MAX)).sum())
panel = df[df.edad.between(EDAD_MIN, EDAD_MAX)].copy()
bitacora.anotar('3.2 Universo', f'Filtro a edad {EDAD_MIN}-{EDAD_MAX} años (universo declarado en Etapa 1.6)',
                n_fuera_edad,
                'La pregunta de investigación está acotada a la edad teórica de 3.er ciclo y educación media; '
                'incluir otras edades mezclaría fenómenos distintos (deserción universitaria, alfabetización adulta, etc.).',
                antes, len(panel))

n_dpto_faltante = int(panel.departamento.isna().sum())
if n_dpto_faltante:
    antes = len(panel)
    panel = panel[panel.departamento.notna()].copy()
    bitacora.anotar('3.2 Universo', 'Eliminación de filas sin departamento mapeable', n_dpto_faltante,
                    'Código de departamento fuera del rango documentado (0-17); no se puede clasificar territorialmente.',
                    antes, len(panel))

print(f'\nUniverso de análisis (12-17 años, 2022-2024): {len(panel):,} personas encuestadas')
print(f'Población representada (ponderada): {panel.factor_expansion.sum():,.0f} personas')

In [ ]:
# ============================================================================
# 3.3 — Construcción de la variable de asistencia y de inasistencia
# ============================================================================
panel['asiste_actualmente'] = (panel.cod_asistencia != COD_NO_ASISTE).astype(int)
panel['no_asiste'] = 1 - panel['asiste_actualmente']
bitacora.anotar('3.3 Derivadas', f'Creación de «asiste_actualmente» / «no_asiste» a partir del código {COD_NO_ASISTE}',
                len(panel), 'ED08 trae 19 códigos de nivel/modalidad para quienes SÍ asisten; se colapsan a un indicador '
                'binario porque la pregunta de investigación es sobre inasistencia, no sobre en qué nivel están matriculados.')

# El motivo (ED10) sólo tiene sentido para quienes no asisten; se documenta el universo de esa variable
con_motivo = panel[panel.no_asiste == 1]
n_motivo_valido = int(con_motivo.motivo_inasistencia.notna().sum())
print(f'Personas que NO asisten (muestra): {len(con_motivo):,}')
print(f'De ellas, con motivo válido registrado: {n_motivo_valido:,} '
      f'({100*n_motivo_valido/max(len(con_motivo),1):.1f} %)')

In [ ]:
# ============================================================================
# 3.4 — Valores atípicos e inconsistencias
# ============================================================================
# El factor de expansión debe ser positivo; un valor negativo o cero sería un error de carga.
incons = [
    ('factor_expansion <= 0', int((panel.factor_expansion <= 0).sum())),
    ('edad fuera de 12-17 (control post-filtro)', int((~panel.edad.between(EDAD_MIN, EDAD_MAX)).sum())),
    ('cod_sexo fuera de {1, 6}', int((~panel.cod_sexo.isin([1, 6])).sum())),
]
display(pd.DataFrame(incons, columns=['Chequeo', 'Registros']))

# Distribución del factor de expansión (para detectar valores extremos que distorsionen ponderaciones)
display(panel.factor_expansion.describe(percentiles=[.01, .05, .5, .95, .99]))

bitacora.anotar('3.4 Atípicos', 'Factores de expansión extremos CONSERVADOS', 0,
                'Los pesos muestrales grandes son parte del diseño (representan estratos con menor tasa de muestreo); '
                'no son errores de carga y eliminarlos introduciría sesgo, no lo corregiría.')

In [ ]:
# ============================================================================
# 3.5 — Funciones de estadística ponderada (usadas en toda la Etapa 4 y 5)
# ============================================================================

def tasa_ponderada(df, col_indicador, col_peso='factor_expansion') -> float:
    """% ponderado de un indicador 0/1, expandido a la población representada."""
    return 100 * np.average(df[col_indicador], weights=df[col_peso])


def tabla_tasas_por_grupo(df, col_grupo, col_indicador='no_asiste', col_peso='factor_expansion') -> pd.DataFrame:
    def _agg(g):
        return pd.Series({
            'n_muestral': len(g),
            'poblacion_ponderada': g[col_peso].sum(),
            'tasa_%': tasa_ponderada(g, col_indicador, col_peso),
        })
    return df.groupby(col_grupo).apply(_agg).sort_values('tasa_%', ascending=False)


def tabla_contingencia_ponderada(df, fila, columna, col_peso='factor_expansion') -> pd.DataFrame:
    """Tabla de doble entrada con SUMA de pesos (no conteo de filas)."""
    return df.pivot_table(index=fila, columns=columna, values=col_peso, aggfunc='sum', fill_value=0)


print('Funciones de estadística ponderada listas: tasa_ponderada(), tabla_tasas_por_grupo(), tabla_contingencia_ponderada()')
print('\nVerificación: tasa nacional de inasistencia (12-17), ponderada, por año de encuesta:')
display(tabla_tasas_por_grupo(panel, 'anio_encuesta').round(2))

In [ ]:
# ============================================================================
# 3.6 — Bitácora y exportación del dataset final
# ============================================================================
tabla_bitacora = bitacora.tabla()
display(tabla_bitacora)
tabla_bitacora.to_csv(DIR_TAB / 'bitacora_limpieza.csv', index=False)

COLS_FINALES = ['anio_encuesta', 'UPM', 'NVIVI', 'NHOGA', 'L02', 'edad', 'sexo', 'zona', 'departamento',
                'sector', 'asiste_actualmente', 'no_asiste', 'motivo_inasistencia', 'factor_expansion']
final = panel[COLS_FINALES].copy()

print(f'Filas iniciales (todas las edades, 2022-2024): {FILAS_INICIALES:,}')
print(f'Filas finales (universo 12-17 años)          : {len(final):,}')
print(f'Población representada (ponderada)           : {final.factor_expansion.sum():,.0f}')

try:
    ruta = DIR_PROC / 'dataset_fase1.parquet'; final.to_parquet(ruta, index=False)
except Exception:
    ruta = DIR_PROC / 'dataset_fase1.csv'; final.to_csv(ruta, index=False, encoding='utf-8')
print('Dataset limpio guardado en:', ruta.relative_to(RAIZ))

dicc_final = diccionario_datos(final)
dicc_final.to_csv(DIR_TAB / 'diccionario_datos_final.csv', index=False)
display(dicc_final)

---

# Etapa 4 — Análisis univariado y bivariado

> **Nota metodológica que atraviesa toda esta etapa:** por tratarse de una encuesta muestral, todas las
> frecuencias, tasas y proporciones se calculan **ponderadas por `factor_expansion`**. Los conteos simples
> de filas (`n_muestral`) se reportan aparte porque indican la precisión de la estimación, no el tamaño
> real de la población.

## 4.1. Univariado — variable numérica (edad)

In [ ]:
# ============================================================================
# Estadísticos de la edad (única variable numérica sustantiva del panel)
# ============================================================================
def resumen_numerico_ponderado(df, col, peso='factor_expansion'):
    valores, pesos = df[col].values, df[peso].values
    media = np.average(valores, weights=pesos)
    varianza = np.average((valores - media)**2, weights=pesos)
    orden = np.argsort(valores)
    v_ord, p_ord = valores[orden], pesos[orden]
    cum = np.cumsum(p_ord) / p_ord.sum()
    percentil = lambda q: v_ord[np.searchsorted(cum, q)]
    return {
        'n': len(df), 'media_ponderada': media, 'desvio_ponderado': np.sqrt(varianza),
        'p5': percentil(.05), 'p25': percentil(.25), 'mediana': percentil(.50),
        'p75': percentil(.75), 'p95': percentil(.95), 'min': valores.min(), 'max': valores.max(),
        'asimetria_muestral': stats.skew(valores), 'curtosis_muestral': stats.kurtosis(valores),
    }

resumen_edad = resumen_numerico_ponderado(final, 'edad')
display(pd.Series(resumen_edad).to_frame('edad (12-17 años)'))

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.histplot(data=final, x='edad', weights='factor_expansion', bins=6, discrete=True, ax=ax, color='#4c72b0')
ax.set_title('Distribución ponderada de la edad en el universo de análisis (12-17 años)')
ax.set_xlabel('Edad (años)'); ax.set_ylabel('Población representada (personas)')
plt.savefig(DIR_FIG / '01_univariado_edad.png'); plt.show()
print('🟡 Con 6 edades exactas (12 a 17), la distribución debería ser casi uniforme si el muestreo es correcto;',
      'describir si se observa algún patrón inesperado.')

## 4.2. Univariado — variables categóricas y concentración

In [ ]:
# ============================================================================
# Frecuencias ponderadas y concentración
# ============================================================================
VARS_CAT = ['zona', 'sexo', 'departamento', 'sector', 'anio_encuesta']
for c in VARS_CAT:
    tab = (final.groupby(c)['factor_expansion'].sum().sort_values(ascending=False))
    tab_pct = (100 * tab / tab.sum()).round(2)
    resumen = pd.concat([tab.round(0).rename('poblacion_ponderada'), tab_pct.rename('%')], axis=1)
    resumen['acumulada_%'] = resumen['%'].cumsum().round(2)
    print(f'\n=== {c} — {tab.size} categorías ===')
    display(resumen)

In [ ]:
# ============================================================================
# Gráfico univariado de zona y sector (ponderados)
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, col in zip(axes, ['zona', 'sector']):
    tab = final.groupby(col)['factor_expansion'].sum().sort_values(ascending=False)
    ax.bar(tab.index.astype(str), tab.values, color='#4c72b0')
    ax.set_title(f'Población de 12-17 años por {col} (ponderada)')
    ax.set_ylabel('Personas representadas'); ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.savefig(DIR_FIG / '02_univariado_categoricas.png'); plt.show()

## 4.3. Bivariado — la variable central: tasa de inasistencia por grupo

In [ ]:
# ============================================================================
# Tasa de inasistencia ponderada por zona, sexo y año
# ============================================================================
for col in ['zona', 'sexo', 'anio_encuesta']:
    print(f'\n=== Tasa de inasistencia (12-17) según {col} ===')
    display(tabla_tasas_por_grupo(final, col).round(2))

In [ ]:
# ============================================================================
# Bivariado numérica-categórica: ¿la edad se asocia con la inasistencia?
# ============================================================================
print('Edad media ponderada, según asiste / no asiste:')
display(final.groupby('no_asiste').apply(
    lambda g: pd.Series({'edad_media_pond': np.average(g.edad, weights=g.factor_expansion), 'n': len(g)})
))

fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(data=final, x='no_asiste', y='edad', ax=ax, inner='quartile', cut=0)
ax.set_xticklabels(['Asiste', 'No asiste'])
ax.set_title('Distribución de edad según condición de asistencia')
ax.set_xlabel(''); ax.set_ylabel('Edad (años)')
plt.savefig(DIR_FIG / '03_edad_por_asistencia.png'); plt.show()
print('🟡 Interpretar: ¿la inasistencia se concentra en las edades más altas del rango (16-17), coherente con',
      'el paso a educación media, o está repartida uniformemente?')

## 4.4. Bivariado — categórica frente a categórica (tablas de contingencia ponderadas)

In [ ]:
# ============================================================================
# Tabla de contingencia ponderada: zona × sector (entre quienes asisten)
# ============================================================================
asisten = final[final.asiste_actualmente == 1]
cont = tabla_contingencia_ponderada(asisten, 'zona', 'sector')
print('--- Población ponderada que asiste, por zona y sector ---'); display(cont.round(0))
print('--- % por fila (dentro de cada zona) ---')
display((100 * cont.div(cont.sum(axis=1), axis=0)).round(2))

chi2, p, gl, esp = stats.chi2_contingency(cont)
v = np.sqrt(chi2 / (cont.values.sum() * (min(cont.shape)-1)))
print(f'\nExploratorio (sobre frecuencias ponderadas, no el n muestral real):',
      f'χ² = {chi2:,.1f} | p = {p:.4g} | V de Cramér = {v:.3f}')
print('⚠️ Este chi-cuadrado usa la población EXPANDIDA como si fuera el tamaño muestral real, lo que infla',
      'artificialmente la significancia. Es válido para describir la asociación (V de Cramér), pero el',
      'contraste formal de la Fase 2 deberá ejecutarse sobre el n muestral real o con métodos de encuestas complejas.')

In [ ]:
# ============================================================================
# Tabla de contingencia ponderada: zona × motivo de inasistencia
# ============================================================================
no_asisten = final[final.no_asiste == 1].dropna(subset=['motivo_inasistencia'])
cont_motivo = tabla_contingencia_ponderada(no_asisten, 'motivo_inasistencia', 'zona')
cont_motivo['total'] = cont_motivo.sum(axis=1)
cont_motivo = cont_motivo.sort_values('total', ascending=False).drop(columns='total')
display(cont_motivo.round(0))
print('\n--- % de cada motivo dentro de cada zona ---')
display((100 * cont_motivo.div(cont_motivo.sum(axis=0), axis=1)).round(1))

## 4.5. Análisis temporal (2022–2024)

In [ ]:
# ============================================================================
# Evolución de la tasa de inasistencia por año y zona
# ============================================================================
evo = final.groupby(['anio_encuesta', 'zona']).apply(
    lambda g: tasa_ponderada(g, 'no_asiste')).unstack()
display(evo.round(2))
evo.to_csv(DIR_TAB / 'serie_temporal_tasa_inasistencia.csv')

fig, ax = plt.subplots(figsize=(9, 5))
for col in evo.columns:
    ax.plot(evo.index, evo[col], marker='o', lw=2, label=col)
ax.set_title('Evolución de la tasa de inasistencia escolar (12-17 años) según zona')
ax.set_xlabel('Año de la encuesta'); ax.set_ylabel('Tasa de inasistencia (%, ponderada)')
ax.set_xticks(evo.index); ax.legend(title='Zona')
plt.savefig(DIR_FIG / '04_evolucion_tasa.png'); plt.show()

---

# Etapa 5 — Análisis descriptivo y exploratorio

> Mínimo **ocho visualizaciones distintas**, cada una con título descriptivo, ejes rotulados con unidades,
> leyenda cuando corresponda, escala que no distorsione y una **interpretación escrita de una a tres
> oraciones inmediatamente debajo**. Todas las cifras están ponderadas por el factor de expansión.

In [ ]:
# === G1 · Serie temporal — tasa de inasistencia por zona (2022-2024) ========
fig, ax = plt.subplots(figsize=(10, 5))
for col in evo.columns:
    ax.plot(evo.index, evo[col], marker='o', lw=2.2, label=col)
ax.set_title('G1 · Evolución de la tasa de inasistencia escolar (12-17 años), por zona')
ax.set_xlabel('Año de la encuesta'); ax.set_ylabel('Tasa de inasistencia (%, ponderada)')
ax.set_xticks(evo.index); ax.legend(title='Zona')
plt.savefig(DIR_FIG / 'G1_serie_tasa_zona.png'); plt.show()

> 🟡 **Interpretación G1:** _describir si la brecha urbano/rural se mantiene, se amplía o se cierra en el período, y comentar la magnitud de la diferencia en puntos porcentuales._

In [ ]:
# === G2 · Histograma — distribución de edad en el universo de análisis =====
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.histplot(data=final, x='edad', weights='factor_expansion', bins=6, discrete=True, ax=ax, color='#4c72b0')
ax.set_title('G2 · Distribución ponderada de la edad, personas de 12 a 17 años (2022-2024)')
ax.set_xlabel('Edad (años)'); ax.set_ylabel('Población representada (personas)')
plt.savefig(DIR_FIG / 'G2_histograma_edad.png'); plt.show()

> 🟡 **Interpretación G2:** _comentar si la distribución por edad simple es razonablemente uniforme (esperable en un universo de 6 edades consecutivas) y si hay algún año con menos representación relativa._

In [ ]:
# === G3 · Diagrama de caja/violín — edad según condición de asistencia =====
fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=final, x='no_asiste', y='edad', ax=ax, inner='quartile', cut=0, palette=['#4c72b0', '#c44e52'])
ax.set_xticklabels(['Asiste', 'No asiste'])
ax.set_title('G3 · Distribución de edad según condición de asistencia actual')
ax.set_xlabel(''); ax.set_ylabel('Edad (años)')
plt.savefig(DIR_FIG / 'G3_edad_asistencia.png'); plt.show()

> 🟡 **Interpretación G3:** _¿la inasistencia se concentra en las edades más próximas a la educación media (16-17 años)? Vincular con la transición de nivel educativo._

In [ ]:
# === G4 · Barras comparativas — tasa de inasistencia por departamento ======
tab_dpto = tabla_tasas_por_grupo(final, 'departamento')
fig, ax = plt.subplots(figsize=(10, 7))
colores = ['#c44e52' if n < 150 else '#4c72b0' for n in tab_dpto['n_muestral']]
ax.barh(tab_dpto.index[::-1], tab_dpto['tasa_%'][::-1], color=colores[::-1])
ax.axvline(tasa_ponderada(final, 'no_asiste'), color='black', ls='--', lw=1.3,
           label=f'Media nacional ({tasa_ponderada(final, "no_asiste"):.1f} %)')
ax.set_title('G4 · Tasa de inasistencia escolar (12-17 años) por departamento, 2022-2024')
ax.set_xlabel('Tasa de inasistencia (%, ponderada)'); ax.set_ylabel('Departamento')
ax.legend()
ax.text(.98, .02, 'En rojo: departamentos con n muestral < 150 (estimación menos precisa)',
        transform=ax.transAxes, ha='right', fontsize=8, style='italic')
plt.tight_layout(); plt.savefig(DIR_FIG / 'G4_barras_departamento.png'); plt.show()
display(tab_dpto.round(2))

> 🟡 **Interpretación G4:** _identificar los departamentos con mayor y menor tasa, y señalar cuáles de esos extremos son estadísticamente menos confiables por bajo n muestral._

In [ ]:
# === G5 · Mapa de calor — motivo de inasistencia según zona =================
cont_motivo_pct = (100 * cont_motivo.div(cont_motivo.sum(axis=0), axis=1))
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cont_motivo_pct, annot=True, fmt='.1f', cmap='Reds', ax=ax,
            cbar_kws={'label': '% dentro de la zona'})
ax.set_title('G5 · Motivo de inasistencia declarado, según zona (% dentro de cada zona)')
ax.set_xlabel('Zona'); ax.set_ylabel('Motivo declarado')
plt.tight_layout(); plt.savefig(DIR_FIG / 'G5_heatmap_motivo_zona.png'); plt.show()

> 🟡 **Interpretación G5:** _señalar si el motivo principal difiere entre zonas (por ejemplo, si en zona rural pesa más 'no existe institución cercana' y en zona urbana 'no quiere estudiar' o 'necesidad de trabajar')._

In [ ]:
# === G6 · Barras — top motivos de inasistencia a nivel nacional ============
motivo_nac = (no_asisten.groupby('motivo_inasistencia')['factor_expansion'].sum()
              .sort_values(ascending=False))
motivo_nac_pct = (100 * motivo_nac / motivo_nac.sum())
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(motivo_nac_pct.index[::-1], motivo_nac_pct.values[::-1], color='#dd8452')
ax.set_title('G6 · Motivos declarados de inasistencia escolar, 12-17 años (nacional, 2022-2024)')
ax.set_xlabel('Participación en el total de inasistentes (%, ponderada)'); ax.set_ylabel('')
plt.tight_layout(); plt.savefig(DIR_FIG / 'G6_motivos_nacional.png'); plt.show()

> 🟡 **Interpretación G6:** _describir el ranking de motivos y agruparlos conceptualmente (económicos, familiares, de oferta educativa, de interés personal) para preparar la interpretación de política. _

In [ ]:
# === G7 · Barras agrupadas — sector educativo por zona (entre quienes asisten) ===
cont_pct = (100 * cont.div(cont.sum(axis=1), axis=0))
ax = cont_pct.plot(kind='bar', figsize=(9, 5.5), width=.7)
ax.set_title('G7 · Composición por sector educativo, según zona (entre quienes asisten)')
ax.set_xlabel('Zona'); ax.set_ylabel('% dentro de la zona (ponderado)')
ax.legend(title='Sector'); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(DIR_FIG / 'G7_sector_por_zona.png'); plt.show()

> 🟡 **Interpretación G7:** _comparar la participación del sector privado y privado subvencionado entre zonas, y vincularlo con la disponibilidad de oferta no oficial en el área rural._

In [ ]:
# === G8 · Dispersión — tasa de inasistencia vs. tamaño de la submuestra por departamento ===
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=tab_dpto.reset_index(), x='n_muestral', y='tasa_%', s=90, ax=ax, color='#4c72b0')
for _, fila in tab_dpto.reset_index().iterrows():
    ax.annotate(fila['departamento'], (fila['n_muestral'], fila['tasa_%']), fontsize=8,
                xytext=(4, 4), textcoords='offset points')
ax.axhline(tasa_ponderada(final, 'no_asiste'), color='black', ls='--', lw=1, alpha=.6)
ax.set_title('G8 · Tasa de inasistencia por departamento frente al tamaño de su submuestra')
ax.set_xlabel('Casos muestrales en el departamento (n)'); ax.set_ylabel('Tasa de inasistencia (%, ponderada)')
plt.tight_layout(); plt.savefig(DIR_FIG / 'G8_dispersion_n_tasa.png'); plt.show()

> 🟡 **Interpretación G8:** _este gráfico es una autocrítica metodológica: verificar si las tasas más extremas (muy altas o muy bajas) corresponden justamente a los departamentos con menos casos muestrales, lo que obligaría a interpretarlas con cautela en la Fase 2._

## 5.2. Patrones y anomalías identificados

🟡 **COMPLETAR — mínimo tres, cada uno con la evidencia que lo respalda, citada por su número de gráfico o tabla.**

| N.° | Patrón o anomalía | Evidencia | Lectura |
|:--:|---|---|---|
| **P1** | *(ej.)* La tasa de inasistencia rural más que duplica a la urbana en los tres años | G1, tabla 4.3 | Brecha estructural, no un episodio de un año |
| **P2** | *(ej.)* El motivo económico/familiar concentra la mayoría de los casos en zona rural, mientras que 'no quiere estudiar' pesa más en zona urbana | G5, G6 | El diseño de política debería diferenciarse por zona |
| **P3** | *(ej.)* Los departamentos con tasas más extremas tienden a ser los de menor n muestral | G8 | Advertencia de precisión: no sobre-interpretar esos extremos sin más evidencia |
| **A1** | *(ej.)* El departamento X muestra una tasa muy por encima del resto | G4 | ¿Hallazgo real o artefacto de muestra pequeña? A verificar en Fase 2 |

## 5.3. Hipótesis preliminares para la Fase 2

> Este es el **puente entre fases** y su ausencia se penaliza. Cada hipótesis deriva de algo observado
> arriba. La prueba definitiva depende de la verificación de supuestos en la Fase 2, y deberá decidirse si
> se trabaja sobre el n muestral real, sobre la población ponderada, o con métodos específicos de encuestas
> complejas (a discutir con el docente).

| N.° | Hipótesis (lenguaje natural) | Evidencia que la motiva | Prueba prevista | Familia | Tamaño de efecto |
|:--:|---|---|---|---|---|
| **H1** | La tasa de inasistencia es mayor en zona rural que en zona urbana. | G1, tabla 4.3 | **Prueba z de proporciones** (o chi-cuadrado 2×2) sobre el n muestral real | Dos proporciones | h de Cohen o diferencia de proporciones con IC |
| **H2** | Existe asociación entre la zona y el motivo declarado de inasistencia (agrupado en categorías: económico, familiar, de oferta, personal). | G5, G6 | **Chi-cuadrado de independencia** (Fisher si hay categorías con frecuencia baja) | Categórica × categórica | V de Cramér |
| **H3** | La tasa de inasistencia difiere entre departamentos. | G4 | **Chi-cuadrado de independencia** (departamento × asiste/no asiste) o **ANOVA/Kruskal-Wallis** sobre proporciones por conglomerado | Tres o más grupos | eta cuadrado o V de Cramér |
| **H4** | La tasa de inasistencia aumenta con la edad dentro del rango 12-17 años. | G3 | Prueba sobre **rho de Spearman** entre edad y condición de inasistencia, o **regresión logística simple** | Correlación / asociación ordinal | rho o razón de momios |
| **H5** | La diferencia urbano/rural en inasistencia persiste al controlar por departamento. | P3 sugiere que zona y departamento pueden confundirse | **Regresión logística múltiple**: inasistencia ~ zona + departamento + edad + sexo | Efecto controlando otras | razón de momios (OR) por variable |

**Cobertura de familias:** H1 (dos proporciones) · H2 (asociación categórica) · H3 (tres o más grupos) · H4
(correlación/asociación ordinal) · H5 (regresión múltiple) → **cinco familias distintas**, por encima del
mínimo de tres exigido.

**Previsiones para la Fase 2**

- α = 0,05 declarado **antes** de ejecutar cualquier prueba.
- Corrección de **Benjamini-Hochberg** sobre la familia H1–H4, por tratarse de contrastes múltiples sobre el
  mismo fenómeno.
- **Decisión metodológica a resolver con el docente:** si los contrastes formales se ejecutan sobre el **n
  muestral real** (más conservador, pero ignora el diseño de ponderación) o replicando el peso muestral con
  técnicas de remuestreo. Se documentará la elección y su justificación al inicio de la Fase 2.
- **Confusor principal a discutir:** zona y departamento pueden estar correlacionados (algunos departamentos
  son mayoritariamente rurales); H5 existe justamente para separar ambos efectos.

---

# 6. Conclusión de la Fase 1

> Sección obligatoria de **una a dos páginas**. 🟡 Redactar en prosa a partir de los resultados que
> efectivamente arrojó este notebook. El esqueleto de abajo es la estructura exigida, no el texto final.

### 6.1. Principales resultados obtenidos

_Síntesis de la caracterización: magnitud de la tasa de inasistencia nacional y su evolución 2022-2024,
tamaño de la brecha urbano/rural, jerarquía de motivos declarados y variación entre departamentos._

### 6.2. Respuesta parcial a la pregunta de investigación

_Qué se puede afirmar **sólo con el análisis descriptivo**: que existen diferencias observadas en tal
magnitud y dirección entre zonas y departamentos, y que los motivos declarados siguen tal jerarquía. Y qué
**todavía no** se puede afirmar: si esas diferencias son estadísticamente sostenibles una vez controladas
otras variables (Fase 2), y si permiten anticipar el riesgo de inasistencia de un adolescente concreto
(Fase 3)._

### 6.3. Limitaciones detectadas en los datos

_Las efectivamente encontradas: departamentos con submuestra pequeña y estimaciones menos precisas (G8);
el hecho de no contar con características del establecimiento; la naturaleza de corte transversal repetido
(no panel) que impide seguir a las mismas personas en el tiempo._

### 6.4. Decisiones que condicionan las fases siguientes

_Cada decisión de la Etapa 3 con su consecuencia. En particular: el uso obligatorio del factor de expansión,
la decisión pendiente sobre qué unidad usar en los contrastes formales de la Fase 2 (ponderada vs. muestral),
y que la ausencia de identificación del establecimiento limita a la Fase 3 a predictores puramente
territoriales y demográficos (zona, departamento, edad, sexo), sin poder incorporar variables institucionales._

### 6.5. Hipótesis que se llevan a la Fase 2

_Las cinco de la sección 5.3, con su orden de prioridad, y la variable objetivo elegida para la Fase 3:
`no_asiste` (clasificación binaria)._

---

## 7. Declaración de uso de asistentes de IA

> Requerido por la sección 11 del enunciado. 🟡 Completar con veracidad.

| Tarea | ¿Se usó IA? | Detalle |
|---|:--:|---|
| Definición de la pregunta y de los objetivos | | |
| Estructura del notebook y del informe | | |
| Escritura y depuración de código | | |
| Selección de pruebas y decisiones metodológicas | | |
| Interpretación de resultados y conclusiones | | |
| Redacción del informe | | |

El grupo declara haber **verificado y comprendido** todo el código y todo el texto incorporado, y estar en
condiciones de responder sobre cualquier línea del trabajo durante la presentación final.

In [ ]:
# ============================================================================
# Verificación automática de los entregables de la Fase 1
# ============================================================================
chequeos = [
    ('1 · Documento de comprensión del problema (PDF 2–3 pp.)', (RAIZ/'output'/'fase1_comprension_problema.pdf').exists()),
    ('2 · Informe de la Fase 1 (PDF ≤ 20 pp.)',                 (RAIZ/'output'/'Informe_Fase1.pdf').exists()),
    ('3 · Notebook ejecutado de principio a fin sin errores',   True),
    ('3b · Exportación HTML del notebook',                      any(RAIZ.rglob('*.html'))),
    ('4 · Diccionario de datos original',                       (DIR_TAB/'diccionario_datos_original.csv').exists()),
    ('4b · Diccionario de datos final',                         (DIR_TAB/'diccionario_datos_final.csv').exists()),
    ('5 · Dataset limpio',                                      any(DIR_PROC.glob('dataset_fase1.*'))),
    ('6 · README de reproducción',                              (RAIZ/'README.md').exists()),
    ('6b · Archivo de dependencias',                            (RAIZ/'requirements.txt').exists()),
    ('· Bitácora de limpieza',                                  (DIR_TAB/'bitacora_limpieza.csv').exists()),
    ('· Ocho visualizaciones generadas',                        len(list(DIR_FIG.glob('G*.png'))) >= 8),
    ('· Serie temporal 2022-2024 incluida',                     (DIR_TAB/'serie_temporal_tasa_inasistencia.csv').exists()),
]
for nombre, ok in chequeos:
    print(f'{"✅" if ok else "⬜"}  {nombre}')
pend = [n for n, ok in chequeos if not ok]
print(f'\n{len(chequeos)-len(pend)}/{len(chequeos)} listos.')
if pend:
    print('Pendientes (dependen del grupo, no del código):')
    for p in pend:
        print('   •', p)